<a href="https://colab.research.google.com/github/aishnori-abhyasi/AI-Product-Leader_Capstone/blob/main/RAG_in_Action.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialisation

In [1]:
import nltk
from nltk import word_tokenize
from nltk import sent_tokenize

from nltk import pos_tag
from nltk.tag.mapping import map_tag

import numpy as np

nltk.download('all', quiet=True)

True

In [2]:
def get_upos_tags(text: str):
    """
    Tokenizes the text and returns (word, UPOS) pairs.
    Example UPOS tags: NOUN, VERB, ADJ, DET, ADP, etc.
    """
    word_list = word_tokenize(text)
    treebank_tags = pos_tag(word_list)
    upos_tags = [(word, map_tag('en-ptb', 'universal', tag)) for word, tag in treebank_tags]
    return upos_tags

In [3]:
# Install the Sentence Transformer library
!pip install --quiet --upgrade sentence-transformers

import os
import logging

# Suppress Hugging Face / Transformers logs
os.environ["TRANSFORMERS_VERBOSITY"] = "error"  # only show errors

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

from sentence_transformers import SentenceTransformer, util

# Load model silently
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 34.7 MB/s eta 0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
def get_embedding(text):
    return model.encode(text,convert_to_tensor=True)

def find_text_similarity(text1, text2):
    return round(float(util.pytorch_cos_sim(get_embedding(text1), get_embedding(text2))[0][0]), 4)

def find_embedding_similarity(emb1, emb2):
    return round(float(util.pytorch_cos_sim(emb1, emb2)[0][0]), 4)

In [5]:
def create_embedding_list(text_list):
    return [get_embedding(text) for text in text_list]

In [6]:
def find_answer(query, text_list, embedding_list):
    query_embedding = get_embedding(query)
    sim_list = [find_embedding_similarity(query_embedding, emb) for emb in embedding_list]
    print("Similarity values between query and all sentences in the list:")
    print(sim_list)
    print("")
    return text_list[sim_list.index(max(sim_list))]

In [7]:
import requests
from bs4 import BeautifulSoup

def fetch_wikipedia_html(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/91.0.4472.114 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.text

def extract_paragraphs(html: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")

    # Main content of Wikipedia pages is inside <div id="mw-content-text">
    content_div = soup.find("div", {"id": "mw-content-text"})
    paragraphs = []

    if content_div:
        for p in content_div.find_all("p"):
            text = p.get_text().strip()
            if text:  # ignore empty or citation-only paragraphs
                paragraphs.append(text)
    return paragraphs

In [8]:
def find_answer_top_k(query, text_list, embedding_list, k=2):
    query_embedding = get_embedding(query)
    sim_list = [find_embedding_similarity(query_embedding, emb) for emb in embedding_list]

    # Get indices of top-k similarities (sorted descending)
    top_k_indices = np.argsort(sim_list)[::-1][:k]

    # Return top-k texts as a list
    return [text_list[i] for i in top_k_indices]


In [9]:
from openai import OpenAI

from dotenv import load_dotenv
import os
# load_dotenv("api_key.env")
# api_key = os.getenv('OPENROUTER_API_KEY')
api_key = "PASTE YOUR API KEY HERE"

# 1. Visit : https://openrouter.ai/
# 2. Login with your Gmail ID and create a new API Key (top right menu)
# 3. Create api_key.env file in the same folder as your code
# 4. Put the api_key there. Just write one line (use your own API Key, the one below wont work):
# api_key=sk-or-v1-aasdewr34asdc0r31oweijf

def get_ai_response(prompt):
  client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
  )

  completion = client.chat.completions.create(
      model="openai/gpt-oss-20b:free",
    messages=[
      {
        "role": "user",
        "content": prompt
      }
    ]
  )

  return completion.choices[0].message.content


# NLP Tools

In [10]:
text = "I am learning Large Language Models. I will then build a cool application using it."

In [11]:
# Sentence Tokenization
# Tokenization is basically breaking a piece of text
# into smaller units.
sent_tokenize(text)

['I am learning Large Language Models.',
 'I will then build a cool application using it.']

In [12]:
# Word Tokenization
word_tokenize(text)

# What AI does is sub-word tokenization.

['I',
 'am',
 'learning',
 'Large',
 'Language',
 'Models',
 '.',
 'I',
 'will',
 'then',
 'build',
 'a',
 'cool',
 'application',
 'using',
 'it',
 '.']

In [13]:
# POS Tagging (Parts of Speech)
print(get_upos_tags(text))

# One application is sentiment analysis.

# Named Entity Recognition [NER]
# classify words into categories..
# Company Name, Person Name, Brand, Location

[('I', 'PRON'), ('am', 'VERB'), ('learning', 'VERB'), ('Large', 'NOUN'), ('Language', 'NOUN'), ('Models', 'NOUN'), ('.', '.'), ('I', 'PRON'), ('will', 'VERB'), ('then', 'ADV'), ('build', 'VERB'), ('a', 'DET'), ('cool', 'ADJ'), ('application', 'NOUN'), ('using', 'VERB'), ('it', 'PRON'), ('.', '.')]


# LLM Tools

In [14]:
text = "I am a human being."
get_embedding(text)

tensor([ 5.2786e-03,  1.9185e-02,  6.3271e-02,  1.1727e-02, -5.7958e-02,
        -7.7475e-02,  1.4263e-01, -3.2002e-02,  4.6559e-02,  9.9696e-03,
        -9.2031e-03, -9.0321e-02, -3.2316e-02, -1.8097e-02,  7.7099e-02,
        -3.1339e-02, -5.2647e-03, -4.3888e-02, -4.0001e-02,  4.4545e-02,
        -2.9328e-02,  4.6565e-02, -1.5779e-02, -2.0519e-02, -4.9357e-02,
        -1.5548e-02, -1.0629e-02, -3.4567e-02,  6.5551e-02, -1.7427e-02,
         1.2922e-02, -6.9742e-02,  9.9495e-02, -1.9190e-02, -3.6043e-02,
         7.4612e-02,  4.4963e-02, -5.4321e-02,  5.6025e-02,  1.8090e-02,
         3.3005e-02, -8.2020e-02, -6.1142e-03,  5.9380e-03,  3.7282e-02,
         5.0122e-02, -2.1457e-02, -4.2981e-02, -4.2478e-02, -5.2496e-02,
        -7.1995e-02,  1.0558e-01, -3.2934e-02,  6.6258e-02, -8.5375e-03,
        -4.0289e-02,  4.9874e-02,  2.4564e-02,  2.1757e-02, -5.7993e-02,
         9.0196e-03,  9.0460e-03, -1.8721e-02,  1.0411e-01,  4.4548e-02,
        -8.3939e-03,  3.7495e-02, -4.1318e-02, -1.7

In [15]:
text1 = "I am a human being."
text2 = "What are you doing?"
find_text_similarity(text1, text2)
# So this value ranges between -1 to +1.

# ONLY TEXT DOES NOT HAVE A NATURAL NUMERIC REPRESENTATION.

0.3407

# RAG : Retrieval Augmented Generation

# Smarter RAG System

### Retrieving Information from PDFs

To use PDF documents in a RAG (Retrieval-Augmented Generation) pipeline, we perform the following steps:
1. **Install a PDF Parser:** We can use popular libraries like `pypdf` or `PyPDF2` to read PDF files.
2. **Extract Text:** Extract plain text page-by-page from the PDF.
3. **Chunk the Text:** Since PDFs can be very long, we break the extracted text down into smaller, meaningful segments (e.g., sentences or paragraphs) so that we can embed and retrieve specific chunks of information.

In [6]:
# 1. Install the pypdf library
!pip install --quiet pypdf

import urllib.request
from pypdf import PdfReader
import nltk
from nltk import sent_tokenize

# Ensure both NLTK tokenizers and their required tabs are downloaded
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# 2. Download a sample PDF with a custom User-Agent header to avoid 403 Forbidden errors
##pdf_url = "https://unec.edu.az/application/uploads/2014/12/pdf-sample.pdf"
##pdf_path = "sample.pdf"

pdf_url = "https://www.un.org/sites/un2.un.org/files/un_card_2026_e.pdf"
pdf_path = "un_card_2026_e.pdf"

req = urllib.request.Request(
    pdf_url,
    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36'}
)

with urllib.request.urlopen(req) as response, open(pdf_path, 'wb') as out_file:
    out_file.write(response.read())

# 3. Read and extract text from the PDF file
reader = PdfReader(pdf_path)
pdf_text = ""
for page in reader.pages:
    text_content = page.extract_text()
    if text_content:
        pdf_text += text_content + "\n"

# 4. Tokenize the extracted text into sentences
pdf_sentences = sent_tokenize(pdf_text)

print(f"Extracted {len(reader.pages)} pages.")
print(f"Segmented into {len(pdf_sentences)} sentence chunks.")
print("\nFirst 3 chunks:")
for i, chunk in enumerate(pdf_sentences[:3]):
    print(f"[{i}]: {chunk.strip()}")

Extracted 2 pages.
Segmented into 1 sentence chunks.

First 3 chunks:
[0]: THE UNITED NATIONS
Assists and protects 136 million people 
fleeing war, famine and persecution
Provides food and assistance to more than 
110 million people in over 120 countries 
and territories
Supplies vaccines for 45% of the world's 
children, helps save 4 million lives a year
Works with 193 nations to keep the global 
temperature rise below 1.5°C 
Keeps peace with 52,000 peacekeepers 
in 11 operations around the world
Tackles the global water crisis affecting 
over 4 billion people worldwide
Protects and promotes human rights globally 
and through 80 treaties/declarations
Coordinates US$ 23 billion appeal 
prioritizing humanitarian needs of 
87 million people
Uses diplomacy to prevent conflict: 
assists some 50 countries a year with 
their elections
Protects over 42 million people with life-
saving sexual and reproductive services
www.un.org • © United Nations • 2026
